In [ ]:
from google.colab import drive
import os, sys

drive.mount('/content/drive')

# SET THIS to wherever you put the ActiPheno folder in your Google Drive 
# 
PROJECT_DIR = '/content/drive/MyDrive/ActiPheno'

os.chdir(PROJECT_DIR)
sys.path.insert(0, os.path.join(PROJECT_DIR, 'src'))

# Check if src/ is empty you forgot to upload the .py files
print('cwd:', os.getcwd())
print('src contents:', os.listdir('src'))

In [ ]:
import glob

cond = sorted(glob.glob('data/condition/*.csv'))
ctrl = sorted(glob.glob('data/control/*.csv'))

print(f'condition: {len(cond)} files  (expect 23)')
print(f'control:   {len(ctrl)} files  (expect 32)')
print(f'scores.csv: {found if os.path.exists(data/scores.csv) else MISSING}')

if len(cond) != 23 or len(ctrl) != 32:
    print('⚠️  wrong counts — check your Drive uploads')
else:
    print('all good, carry on')

In [ ]:
import logging
logging.basicConfig(level=logging.INFO,
                    format='%(asctime)s  %(levelname)-8s  %(message)s',
                    datefmt='%H:%M:%S')

from data_loader import DepresjonLoader

# min_windows=1 keeps any patient with at least one full day of data
# if patients are being skipped the loader will say exactly who and why
loader = DepresjonLoader(data_dir='data', min_windows=1)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# population-level activity distributions
all_dep = np.concatenate([
    np.log1p(r.raw) for r in loader.records.values() if r.label == 1
])
all_ctl = np.concatenate([
    np.log1p(r.raw) for r in loader.records.values() if r.label == 0
])

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.hist(all_dep, bins=60, color='tomato',    alpha=0.6, density=True, label='Depressed')
ax.hist(all_ctl, bins=60, color='steelblue', alpha=0.6, density=True, label='Healthy')
ax.set_xlabel('log1p(activity counts)')
ax.set_ylabel('density')
ax.set_title('Activity distribution — all patients, all minutes')
ax.legend()

ax2 = axes[1]

# mean activity per minute-of-day averaged across all patients in each class
# shows circadian structure at the population level with no individual trace
def mean_circadian(records, label):
    day = np.zeros(1440)
    n   = 0
    for r in records.values():
        if r.label == label:
            full_days = len(r.raw) // 1440
            if full_days == 0:
                continue
            mat  = r.raw[:full_days * 1440].reshape(full_days, 1440)
            day += np.log1p(mat).mean(axis=0)
            n   += 1
    return day / max(n, 1)

dep_circ = mean_circadian(loader.records, 1)
ctl_circ = mean_circadian(loader.records, 0)
ax2.plot(dep_circ, color='tomato',    lw=1.2, alpha=0.85, label='Depressed (mean)')
ax2.plot(ctl_circ, color='steelblue', lw=1.2, alpha=0.85, label='Healthy (mean)')
ax2.set_xlabel('minute of day (0 = midnight)')
ax2.set_ylabel('mean log1p(activity)')
ax2.set_title('Mean circadian profile by class')
ax2.legend()

plt.tight_layout()
plt.savefig('assets/eda_population.png', dpi=150, bbox_inches='tight')
plt.show()
print('saved → assets/eda_population.png')

In [ ]:
import shutil

LOCAL = '/content/ActiPheno_local'
os.makedirs(LOCAL, exist_ok=True)

local_src  = os.path.join(LOCAL, 'src')
local_data = os.path.join(LOCAL, 'data')

if os.path.exists(local_src):
    shutil.rmtree(local_src)
shutil.copytree(os.path.join(PROJECT_DIR, 'src'), local_src)
print('src copied to local SSD')

# data only needs copying once per session as it doesn't change
if not os.path.exists(local_data):
    print('copying data/ to local SSD (one-time, ~30 seconds)...')
    shutil.copytree(os.path.join(PROJECT_DIR, 'data'), local_data)
    print('done')
else:
    print('data/ already on local SSD')

In [ ]:
import subprocess

SEEDS  = [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52]
AGG    = 'mean'   # window-to-patient aggregation: mean | median | frac
EPOCHS = 25

# results go straight to Drive after every fold so nothing is lost
# if Colab disconnects mid-run, re-run this cell and it picks up
# from where it left off
DRIVE_RUNS = os.path.join(PROJECT_DIR, 'runs')
os.makedirs(DRIVE_RUNS, exist_ok=True)

for seed in SEEDS:
    drive_out = os.path.join(DRIVE_RUNS, f'seed{seed}')
    os.makedirs(drive_out, exist_ok=True)
    finished  = os.path.join(drive_out, f'fold_preds_seed{seed}_agg{AGG}.csv')

    if os.path.exists(finished):
        import pandas as pd
        n_done = len(pd.read_csv(finished))
        if n_done >= len(loader.records):
            print(f'seed {seed} — all {n_done} folds done, skipping')
            continue
        print(f'seed {seed} — {n_done}/{len(loader.records)} folds done, resuming...')
    else:
        print(f'seed {seed} — starting fresh')

    cmd = [
        sys.executable,
        os.path.join(LOCAL, 'src', 'train_eval.py'),
        '--data_dir',     os.path.join(LOCAL, 'data'),
        '--out_dir',      drive_out,
        '--use_delta',
        '--pos_weight',   'auto',
        '--augment',
        '--amp',
        '--epochs',       str(EPOCHS),
        '--batch_size',   '32',
        '--num_workers',  '0',    # 0 avoids silent dataloader deadlocks on Colab
        '--val_per_class','2',
        '--patience',     '5',
        '--agg',          AGG,
        '--seed',         str(seed),
        '--max_folds',    '-1',
    ]

    result = subprocess.run(cmd, cwd=LOCAL)

    if result.returncode != 0:
        print(f'seed {seed} exited with code {result.returncode}')
        print('re-run this cell to resume — completed folds will be skipped')
        break

    print(f'seed {seed} done
')

In [ ]:
import glob, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (
    f1_score, matthews_corrcoef, confusion_matrix,
    roc_curve, auc, precision_recall_curve, average_precision_score,
)

AGG        = 'mean'
DRIVE_RUNS = os.path.join(PROJECT_DIR, 'runs')

csvs = sorted(glob.glob(
    os.path.join(DRIVE_RUNS, f'**/fold_preds_*_agg{AGG}.csv'), recursive=True
))
if not csvs:
    print('no CSVs found — run Cell 5 first')
else:
    all_folds = pd.concat([pd.read_csv(c) for c in csvs], ignore_index=True)
    print(f'loaded {len(all_folds)} fold rows across {all_folds["seed"].nunique()} seed(s)\n')

    # per-seed table 
    rows = []
    for seed, grp in all_folds.groupby('seed'):
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            rows.append({
                'seed':     int(seed),
                'folds':    len(grp),
                'accuracy': (grp.patient_true == grp.patient_pred).mean(),
                'f1':       f1_score(grp.patient_true, grp.patient_pred, zero_division=0),
                'mcc':      matthews_corrcoef(grp.patient_true, grp.patient_pred),
            })
    df = pd.DataFrame(rows)
    print(df.to_string(index=False))
    print()
    for col in ['accuracy', 'f1', 'mcc']:
        v = df[col].values
        print(f'  {col:<10}  {v.mean():.4f} ± {v.std():.4f}')
    print()

    # confusion matrix + score distribution 
    cm = confusion_matrix(all_folds.patient_true, all_folds.patient_pred, labels=[0,1])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))

    ax = axes[0]
    im = ax.imshow(cm, cmap='Blues')
    ax.set_title(f'Global patient-level CM  (all seeds, agg={AGG})')
    ax.set_xticks([0,1]); ax.set_xticklabels(['Healthy','Depressed'])
    ax.set_yticks([0,1]); ax.set_yticklabels(['Healthy','Depressed'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('True')
    plt.colorbar(im, ax=ax)
    for i in range(2):
        for j in range(2):
            ax.text(j, i, str(cm[i,j]), ha='center', va='center', fontsize=16,
                    color='white' if cm[i,j] > cm.max()/2 else 'black')

    ax2 = axes[1]
    for label, color, name in [(1,'tomato','Depressed'),(0,'steelblue','Healthy')]:
        scores = all_folds[all_folds.patient_true == label]['patient_score'].values
        ax2.hist(scores, bins=20, alpha=0.65, color=color, label=name)
    ax2.axvline(0.5, color='black', ls='--', lw=1.5, label='threshold')
    ax2.set_xlabel('mean window probability')
    ax2.set_ylabel('count')
    ax2.set_title('Score distributions by true class')
    ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RUNS, 'final_results.png'), dpi=150, bbox_inches='tight')
    plt.savefig('assets/results.png',                          dpi=150, bbox_inches='tight')
    plt.show()

    # ROC + PR curves which aggregate across all seeds
    fig, axes = plt.subplots(1, 2, figsize=(11, 4))

    fpr_g, tpr_g, _ = roc_curve(all_folds.patient_true, all_folds.patient_score)
    roc_auc_g        = auc(fpr_g, tpr_g)
    axes[0].plot(fpr_g, tpr_g, color='steelblue', lw=2,
                 label=f'Aggregate ROC (AUC = {roc_auc_g:.3f})')
    axes[0].plot([0,1],[0,1], 'k--', lw=1)
    axes[0].set_xlabel('False Positive Rate')
    axes[0].set_ylabel('True Positive Rate')
    axes[0].set_title(f'ROC Curve — all seeds (N={len(all_folds)} decisions)')
    axes[0].legend(loc='lower right')

    prec_g, rec_g, _ = precision_recall_curve(all_folds.patient_true, all_folds.patient_score)
    ap_g              = average_precision_score(all_folds.patient_true, all_folds.patient_score)
    base_rate         = all_folds.patient_true.mean()
    axes[1].plot(rec_g, prec_g, color='tomato', lw=2,
                 label=f'Aggregate PR (AP = {ap_g:.3f})')
    axes[1].axhline(base_rate, color='gray', ls='--', lw=1,
                    label=f'Baseline (prevalence = {base_rate:.2f})')
    axes[1].set_xlabel('Recall')
    axes[1].set_ylabel('Precision')
    axes[1].set_title(f'Precision-Recall Curve — all seeds')
    axes[1].legend(loc='upper right')

    plt.tight_layout()
    plt.savefig(os.path.join(DRIVE_RUNS, 'roc_pr.png'), dpi=150, bbox_inches='tight')
    plt.savefig('assets/roc_pr.png',                    dpi=150, bbox_inches='tight')
    plt.show()
    print('saved → assets/results.png  assets/roc_pr.png')

    print('\n--- vs paper baseline ---')
    print('Garcia-Ceja et al. 2018 (handcrafted + SVM):  F1=0.73  MCC=0.44')
    mean_f1  = df['f1'].mean()
    std_f1   = df['f1'].std()
    mean_mcc = df['mcc'].mean()
    std_mcc  = df['mcc'].std()
    n_seeds  = df['seed'].nunique()
    print(f'ActiPheno (1D-CNN + BiLSTM, {n_seeds} seeds):  '
          f'F1={mean_f1:.3f}±{std_f1:.3f}  MCC={mean_mcc:.3f}±{std_mcc:.3f}')
    print(f'Aggregate ROC-AUC: {roc_auc_g:.3f}  AP: {ap_g:.3f}')

In [ ]:
# Post-hoc error analysis (non-clinical)

# All predictions are already finalised above. This cell cross-references
# those predictions against scores.csv (MADRS, subtype, recording length)
# purely for interpretation. None of this metadata was available to the
# model during training or inference.

import os, glob
import pandas as pd
import matplotlib.pyplot as plt

# scores.csv is in the project folder, not committed to git
scores = pd.read_csv('data/scores.csv')
scores.columns = scores.columns.str.strip().str.lower()
scores = scores.rename(columns={'number': 'patient_id'})

DRIVE_RUNS = os.path.join(PROJECT_DIR, 'runs')
csvs = glob.glob(os.path.join(DRIVE_RUNS, '**/fold_preds_*.csv'), recursive=True)
if not csvs:
    raise FileNotFoundError('No fold prediction files found — run Cell 5 first.')

folds = pd.concat([pd.read_csv(f) for f in csvs], ignore_index=True)

# collapse to one row per patient so majority vote across seeds
per_patient = (
    folds.groupby('patient_id')
    .agg(
        patient_true=('patient_true',  'first'),
        patient_pred=('patient_pred',  lambda x: int(round(x.mean()))),
        patient_score=('patient_score', 'mean'),
    )
    .reset_index()
)

merged = per_patient.merge(
    scores[['patient_id', 'madrs1', 'madrs2', 'afftype', 'days']],
    on='patient_id', how='left'
)

afftype_map = {1: 'bipolar II', 2: 'unipolar', 3: 'bipolar I'}
merged['afftype_str'] = merged['afftype'].map(afftype_map).fillna('healthy')

dep = merged[merged.patient_true == 1].copy()
fn  = dep[dep.patient_pred == 0].copy()
tp  = dep[dep.patient_pred == 1].copy()
fp  = merged[(merged.patient_true == 0) & (merged.patient_pred == 1)].copy()

print('=== FALSE NEGATIVES (missed depressed patients) ===')
print(fn[['patient_id', 'patient_score', 'madrs1', 'madrs2', 'afftype_str', 'days']]
      .sort_values('madrs1').to_string(index=False))
print(f'
mean MADRS1 of false negatives : {fn.madrs1.mean():.1f}')
print(f'mean MADRS1 of true positives  : {tp.madrs1.mean():.1f}')

print('
=== FALSE POSITIVES (healthy patients flagged as depressed) ===')
print(fp[['patient_id', 'patient_score', 'afftype_str', 'days']]
      .sort_values('patient_score', ascending=False).to_string(index=False))

print('
=== ACCURACY BY AFFTYPE (depressed patients only) ===')
dep['correct'] = (dep.patient_true == dep.patient_pred).astype(int)
print(dep.groupby('afftype_str')[['correct', 'patient_score']].mean().round(3))

# score distribution plot
plt.figure(figsize=(6, 4))
plt.hist(tp['patient_score'],  bins=12, alpha=0.7, label='True Positives')
if len(fn) > 0:
    plt.hist(fn['patient_score'], bins=12, alpha=0.7, label='False Negatives')
plt.axvline(0.5, linestyle='--', linewidth=1, color='black', label='threshold')
plt.xlabel('Mean patient probability')
plt.ylabel('Count')
plt.title('Patient-Level Score Distribution')
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(DRIVE_RUNS, 'fn_analysis.png'), dpi=150, bbox_inches='tight')
plt.show()
print('saved → runs/fn_analysis.png')